In [15]:
import pandas as pd
import numpy as np
from econml.dml import LinearDML
from sklearn.linear_model import LassoCV, LogisticRegressionCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, \
    GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
import warnings

warnings.filterwarnings('ignore')

# 1. 基础设置与数据读取
file_path = r"填补的数据整合 - 副本.xlsx"
print("正在读取数据...")
df = pd.read_excel(file_path)


正在读取数据...


In [16]:
df.head()

,省份_final,省份代码_final,地级市,地级市代码,年份_final,初次开放数据平台年份,treat（是否开放数据平台虚拟变量）,post（开放数据平台时间虚拟变量）,DID,诉求量,...,细颗粒物(PM2.5)年平均浓度(微克/立方米),空气质量优良天数比例(%),保险业承保额(万元),城镇职工基本养老保险参保人数(人),职工基本医疗保险参保人数(人),失业保险参保人数(人),提供住宿的社会工作机构数(个),养老机构数(个),提供住宿的社会工作机构床位数(张),养老机构床位数(张)
0,北京市,11,北京,110000,2007,2012.0,1,0,0,0,...,63.0,74.2,317119931.0,7065116.0,9135470.0,6215745.0,491.0,557.0,102841.0,104557.0
1,北京市,11,北京,110000,2008,2012.0,1,0,0,0,...,63.0,74.2,343683451.0,8028270.0,9755582.0,6880443.0,499.0,559.0,103758.0,104891.0
2,北京市,11,北京,110000,2009,2012.0,1,0,0,0,...,63.0,74.2,372472063.0,8995673.0,10417787.0,7538444.0,506.0,560.0,104684.0,105227.0
3,北京市,11,北京,110000,2010,2012.0,1,0,0,0,...,63.0,74.2,403672149.0,9954595.0,11124942.0,8183498.0,514.0,562.0,105617.0,105563.0
4,北京市,11,北京,110000,2011,2012.0,1,0,0,0,...,63.0,74.2,437485708.0,10893893.0,11880098.0,8810357.0,522.0,563.0,106559.0,105900.0


In [17]:
# 2. 变量预处理
# A. 对诉求量取对数 log(Y + 1)
y_col = '诉求量_log'
df[y_col] = np.log1p(df['诉求量'])

# B. 处理时间固定效应 (年份在第E列，即索引4)
year_col_name = df.columns[4]
print(f"检测到年份列: {year_col_name}")
year_dummies = pd.get_dummies(df[year_col_name], prefix='year', drop_first=True)
year_dummy_cols = year_dummies.columns.tolist()

# C. 处理地区变量 (地区在第A列/省份)
region_col_name = df.columns[0]
print(f"检测到地区列: {region_col_name}")

检测到年份列: 年份_final
检测到地区列: 省份_final


In [9]:
year_dummies

,year_2008,year_2009,year_2010,year_2011,year_2012,year_2013,year_2014,year_2015,year_2016,year_2017,year_2018,year_2019,year_2020,year_2021,year_2022,year_2023
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5724,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
5725,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
5726,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
5727,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


In [18]:
# D. 确定控制变量 (X)
t_col = 'post（开放数据平台时间虚拟变量）'
all_cols = df.columns.tolist()
original_x_cols = all_cols[18:142]
x_cols = original_x_cols + year_dummy_cols

# 整合数据并剔除缺失值
df_final = pd.concat([df, year_dummies], axis=1)
cols_to_use = [y_col, t_col, region_col_name] + x_cols
df_dml = df_final.dropna(subset=cols_to_use).copy()


第一阶段——全样本基准估计
目标：估计全国层面的平均政策效应（ATE）

做法：

使用全样本数据（所有省份、所有年份）

对Y（诉求量）和T（post虚拟变量）分别用机器学习模型拟合

计算残差后回归，得到ATE

用6种不同模型组合（Lasso、Ridge、RF、GBDT、XGB、LGBM）做模型稳健性检验

你的逻辑：

“如果用不同机器学习模型都能得到相似的ATE，说明结果对模型选择不敏感，结论更可信。”

In [19]:
# 3. 定义全套机器学习模型组合
# 注意：对于 Ridge 分类器，我们使用 LogisticRegressionCV(penalty='l2') 作为等效实现
models_dict = {
    "1. Lasso (L1正则化)": (
        LassoCV(cv=5),
        LogisticRegressionCV(cv=5, penalty='l1', solver='saga', max_iter=1000)
    ),
    "2. Ridge (L2正则化)": (
        RidgeCV(cv=5),
        LogisticRegressionCV(cv=5, penalty='l2', solver='lbfgs', max_iter=1000)
    ),
    "3. RandomForest (随机森林)": (
        RandomForestRegressor(n_estimators=100, max_depth=6, n_jobs=-1, random_state=123),
        RandomForestClassifier(n_estimators=100, max_depth=6, n_jobs=-1, random_state=123)
    ),
    "4. GradientBoosting (GBDT)": (
        GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=123),
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=123)
    ),
    "5. XGBoost (极端梯度提升)": (
        XGBRegressor(n_estimators=100, max_depth=4, n_jobs=-1, random_state=123),
        XGBClassifier(n_estimators=100, max_depth=4, n_jobs=-1, random_state=123, use_label_encoder=False,
                      eval_metric='logloss')
    ),
    "6. LightGBM (轻量梯度提升)": (
        LGBMRegressor(n_estimators=100, max_depth=4, n_jobs=-1, random_state=123, verbose=-1),
        LGBMClassifier(n_estimators=100, max_depth=4, n_jobs=-1, random_state=123, verbose=-1)
    )
}

# ==========================================
# 第一阶段：全样本基准回归 (不分地区)
# ==========================================
print("\n" + "=" * 80)
print("第一阶段：全样本多模型 DML 估计 (控制了时间固定效应)")
print("=" * 80)

Y_all = df_dml[y_col].values
T_all = df_dml[t_col].values
X_all = df_dml[x_cols].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

global_results = []

for model_name, (mod_y, mod_t) in models_dict.items():
    print(f"正在运行全样本模型: {model_name} ...")
    try:
        est = LinearDML(model_y=mod_y, model_t=mod_t, discrete_treatment=True, cv=5, random_state=456)
        est.fit(Y_all, T_all, X=X_all_scaled, W=None)

        ate = est.const_marginal_ate(X_all_scaled).item()
        summary_table = est.summary().tables[1]
        p_val = float(summary_table.data[1][4])
        stderr = float(summary_table.data[1][2])

        # 标记显著性星号
        stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
        global_results.append([model_name, f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])
    except Exception as e:
        print(f"  [!] {model_name} 运行失败: {e}")
        global_results.append([model_name, "Error", "Error", "Error"])

print("\n【全样本基准结果汇总】")
print(tabulate(global_results, headers=["模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))


第一阶段：全样本多模型 DML 估计 (控制了时间固定效应)
正在运行全样本模型: 1. Lasso (L1正则化) ...
正在运行全样本模型: 2. Ridge (L2正则化) ...
正在运行全样本模型: 3. RandomForest (随机森林) ...
正在运行全样本模型: 4. GradientBoosting (GBDT) ...
正在运行全样本模型: 5. XGBoost (极端梯度提升) ...
正在运行全样本模型: 6. LightGBM (轻量梯度提升) ...

【全样本基准结果汇总】
+----------------------------+-----------------+----------+-------+
| 模型组合                   | ATE (log效应)   |   标准误 |   P值 |
+============================+=================+==========+=======+
| 1. Lasso (L1正则化)        | -1.6151***      |    0.367 | 0     |
+----------------------------+-----------------+----------+-------+
| 2. Ridge (L2正则化)        | -142.2678***    |   35.725 | 0     |
+----------------------------+-----------------+----------+-------+
| 3. RandomForest (随机森林) | 0.9153***       |    0.163 | 0     |
+----------------------------+-----------------+----------+-------+
| 4. GradientBoosting (GBDT) | 1.1241***       |    0.381 | 0.003 |
+----------------------------+-----------------+----------+-------+
| 5. XGBoost

第二阶段——分地区异质性分析
目标：探索政策效应在不同省份之间的差异

做法：

对每个省份单独跑一遍DML

记录每个省份的ATE、标准误、P值

观察哪些省份效应显著、哪些不显著

你的逻辑：

“政策效果可能不是均匀的。通过分省份分析，可以找出政策在哪类地区更有效，为精准施策提供依据。”

In [20]:
# ==========================================
# 第二阶段：分地区异质性分析
# ==========================================
print("\n\n" + "=" * 80)
print("第二阶段：分地区异质性分析")
print("=" * 80)

unique_regions = df_dml[region_col_name].unique()
print(f"共涉及 {len(unique_regions)} 个地区...\n")

regional_results = []

for region in unique_regions:
    df_sub = df_dml[df_dml[region_col_name] == region]

    # 样本量过滤：由于加入了大量时间虚拟变量，子样本量要求略高
    if len(df_sub) < 50:
        continue

    print(f"正在分析地区: {region} (有效样本量: {len(df_sub)})")

    Y_sub = df_sub[y_col].values
    T_sub = df_sub[t_col].values
    X_sub = df_sub[x_cols].values

    scaler_sub = StandardScaler()
    X_sub_scaled = scaler_sub.fit_transform(X_sub)

    for model_name, (mod_y, mod_t) in models_dict.items():
        try:
            est = LinearDML(model_y=mod_y, model_t=mod_t, discrete_treatment=True, cv=5, random_state=456)
            est.fit(Y_sub, T_sub, X=X_sub_scaled, W=None)

            ate = est.const_marginal_ate(X_sub_scaled).item()
            summary_table = est.summary().tables[1]
            p_val = float(summary_table.data[1][4])
            stderr = float(summary_table.data[1][2])

            stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
            regional_results.append([region, model_name, f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])

        except Exception as e:
            # 某些高级树模型在小样本下可能会报错（如叶子节点分裂失败），捕获异常保证循环继续
            pass

print("\n【分地区异质性结果汇总】")
print(tabulate(regional_results, headers=["地区", "模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))
print("\n注：*** p<0.01, ** p<0.05, * p<0.1。ATE 为对数处理后的效应值。")



第二阶段：分地区异质性分析
共涉及 31 个地区...

正在分析地区: 河北省 (有效样本量: 187)
正在分析地区: 山西省 (有效样本量: 187)
正在分析地区: 内蒙古自治区 (有效样本量: 153)
正在分析地区: 辽宁省 (有效样本量: 238)
正在分析地区: 吉林省 (有效样本量: 136)
正在分析地区: 黑龙江省 (有效样本量: 204)
正在分析地区: 江苏省 (有效样本量: 221)
正在分析地区: 浙江省 (有效样本量: 187)
正在分析地区: 安徽省 (有效样本量: 272)
正在分析地区: 福建省 (有效样本量: 153)
正在分析地区: 江西省 (有效样本量: 187)
正在分析地区: 山东省 (有效样本量: 272)
正在分析地区: 河南省 (有效样本量: 289)
正在分析地区: 湖北省 (有效样本量: 204)
正在分析地区: 湖南省 (有效样本量: 221)
正在分析地区: 广东省 (有效样本量: 357)
正在分析地区: 广西壮族自治区 (有效样本量: 238)
正在分析地区: 海南省 (有效样本量: 68)
正在分析地区: 四川省 (有效样本量: 306)
正在分析地区: 贵州省 (有效样本量: 102)
正在分析地区: 云南省 (有效样本量: 136)
正在分析地区: 西藏自治区 (有效样本量: 102)
正在分析地区: 陕西省 (有效样本量: 170)
正在分析地区: 甘肃省 (有效样本量: 204)
正在分析地区: 宁夏回族自治区 (有效样本量: 85)
正在分析地区: 新疆维吾尔自治区 (有效样本量: 68)
CATE Intercept Results:  float division by zero

【分地区异质性结果汇总】
+------------------+----------------------------+--------------------+-----------+-------+
| 地区             | 模型组合                   | ATE (log效应)      |    标准误 |   P值 |
+==================+============================+====================+=

局限	说明	严重程度\
没有平行趋势检验	无法证明政策前处理组和对照组趋势一致	🔴 致命\
post定义可能错误	错时政策（A省2010、B省2020）被当成同时政策	🔴 致命\
没有省份固定效应	无法控制省份间不随时间变化的固有差异	🟠 严重\
分省份没有对照组	单个省份的前后对比，不是真正的DID	🟠 严重\
滞后效应未考虑	政策效果可能不是即时的	🟡 中等\
样本量问题	分省份后样本量小，机器学习易过拟合	🟡 中等\

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# ==========================================
# 1. 读取数据
# ==========================================
file_path = r"【0405】左连结数据整合 - 副本.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

In [2]:
# ==========================================
# 2. 面板数据缺失值处理（按地级市分组插值）
# ==========================================
# 提取需要插值的列：诉求量(被解释变量) 和 R列到AH列(控制变量基础数据)
# 这里以列名或列索引来提取，假设17:34对应R到AJ列
target_cols = ['诉求量'] + df.columns[17:36].tolist()
# 按照地级市分组，使用线性插值处理时间序列上的缺失值
# limit_direction='both' 确保时间序列头尾的缺失值也能向前或向后填充
df[target_cols] = df.groupby('地级市')[target_cols].apply(
    lambda x: x.interpolate(method='linear', limit_direction='both')
).reset_index(level=0, drop=True)

#df.to_excel("1_插值补充后数据.xlsx", index=False)

In [4]:
df.head()

,省份_final,省份代码_final,地级市,地级市代码,年份_final,初次开放数据平台年份,treat（是否开放数据平台虚拟变量）,post（开放数据平台时间虚拟变量）,DID,诉求量,...,生活垃圾无害化处理率(%),人力资本水平,传统基础设施,互联网发展水平,金融发展水平,财政压力水平,科学支出水平,城市发展水平,外商投资水平,交通便捷程度
0,北京市,11,北京,110000,2007,2012.0,1,0,0,0,...,95.73,468.057135,9.791046,15.663835,1.833686,1.105092,13.718362,10.971709,6.976348,0.966817
1,北京市,11,北京,110000,2008,2012.0,1,0,0,0,...,97.71,442.850329,9.835744,15.835903,1.838388,1.066380,13.930523,11.051350,7.029088,0.902412
2,北京市,11,北京,110000,2009,2012.0,1,0,0,0,...,98.22,463.268664,9.839162,15.975086,2.004852,1.144344,14.049058,11.162687,6.926577,0.941541
3,北京市,11,北京,110000,2010,2012.0,1,0,0,0,...,96.95,459.395770,9.912695,16.182788,1.996307,1.154375,14.397254,11.237738,6.891626,0.932581
4,北京市,11,北京,110000,2011,2012.0,1,0,0,0,...,98.24,452.799906,10.055221,15.288634,1.956155,1.079484,14.420234,11.310295,6.570883,0.917912


In [5]:
# ==========================================
# 3. 构造文献要求的控制变量
# ==========================================
# 注意：等号右侧的 ['列名'] 需要你对照生成的 "1_插值补充后数据.xlsx" 里的实际基础表头进行替换
df['人力资本水平'] = df['普通高等学校在校学生数(人)'] / df['户籍人口(万人)']
df['传统基础设施'] = np.log(df['公路货运量(万吨)'] + 1) # 加1防止出现log(0)
df['互联网发展水平'] = np.log(df['电信业务收入(万元)'] + 1)
df['金融发展水平'] = df['年末金融机构各项贷款余额(万元)'] / df['地区生产总值(万元)']
df['财政压力水平'] = df['地方财政一般预算内支出(万元)'] / df['地方财政一般预算内收入(万元)']
df['科学支出水平'] = np.log(df['科学支出(万元)'] + 1)
df['城市发展水平'] = np.log(df['人均地区生产总值(元)'])
df['外商投资水平'] = np.log(df['外商投资企业数(个)'])
df['交通便捷程度'] = df['高速公路里程(公里)'] / df['户籍人口(万人)']
# 将构造好的控制变量放入列表
controls = ['地区生产总值(万元)','人口密度(人／平方公里)', '第三产业增加值占GDP比重(%)', '人力资本水平', '每百人公共图书馆藏书(册、件)',
            '传统基础设施', '互联网发展水平', '金融发展水平', '财政压力水平', '科学支出水平','城市发展水平','外商投资水平','生活垃圾无害化处理率(%)','交通便捷程度']

# 剔除在构造变量后仍然存在缺失值（如缺乏某城市整体面积数据导致无法插值）的样本
df.dropna(subset=['诉求量', 'post（开放数据平台时间虚拟变量）'] + controls, inplace=True)
#df.to_excel("2_处理后含控制变量数据.xlsx", index=False)


In [6]:
# ==========================================
# 4. 双重机器学习 (DML) 基准回归
# ==========================================
df['诉求量_log'] = np.log1p(df['诉求量'])
Y = df['诉求量_log'].values
T = df['post（开放数据平台时间虚拟变量）'].values
X = df[controls].values

# 提取城市与年份固定效应，转化为虚拟变量矩阵，作为混杂因素 W
# DML第一阶段会利用机器学习算法剥离W对Y和T的影响，从而获得纯净的因果效应
W_df = pd.get_dummies(df[['地级市', '年份_final']], columns=['地级市', '年份_final'], drop_first=True)
W = W_df.values

# 初始化 DML 模型，主模型和倾向得分模型均采用随机森林
# 这与你之前使用 DML 测度公共数据开放效应的方法论一致
est = LinearDML(model_y=RandomForestRegressor(n_estimators=100, random_state=42),
                model_t=RandomForestClassifier(n_estimators=100, random_state=42),
                discrete_treatment=True,
                cv=5,
                random_state=42)

print("正在拟合 DML 基准回归模型，可能需要几分钟时间...")
est.fit(Y, T, X=X, W=W)

# 提取回归摘要并导出
summary_df = est.summary().tables[0]
#pd.DataFrame(summary_df.data).to_excel("3_DML基准回归结果.xlsx", index=False, header=False)

正在拟合 DML 基准回归模型，可能需要几分钟时间...


In [7]:
summary_df

,point_estimate,stderr,zstat,pvalue,ci_lower,ci_upper
X0,-0.0,0.0,-0.194,0.846,-0.0,0.0
X1,-0.0,0.0,-1.117,0.264,-0.001,0.0
X2,0.005,0.008,0.606,0.545,-0.01,0.02
X3,-0.0,0.0,-0.599,0.549,-0.001,0.0
X4,0.001,0.001,1.741,0.082,-0.0,0.002
X5,0.012,0.002,6.837,0.0,0.009,0.016
X6,-0.122,0.0,-340.31,0.0,-0.123,-0.121
X7,0.047,0.002,24.853,0.0,0.043,0.051
X8,0.046,0.004,11.006,0.0,0.038,0.054
X9,-0.072,0.001,-67.871,0.0,-0.075,-0.07


In [ ]:
# ==========================================
# 5. 平行趋势检验 (事件研究法)
# ==========================================
# DML中进行平行趋势相对复杂，这里采用经典的加入双向固定效应的事件研究法
# 假设你的数据中存在标明该城市实施政策时间的列 '初次开放数据平台年份'
# 如果部分城市未开放，该列可能为空，需做替换（如替换为9999并过滤）
df_treated = df[df['初次开放数据平台年份'].notna()].copy()
df_treated['相对时间'] = df_treated['年份_final'] - df_treated['初次开放数据平台年份']

# 生成相对年份的虚拟变量，以政策实施前一期（time_-1）作为基准组，避免完全多重共线性
time_dummies = pd.get_dummies(df_treated['相对时间'], prefix='time', dtype=int)
if 'time_-1.0' in time_dummies.columns:
    time_dummies.drop(columns=['time_-1.0'], inplace=True)
elif 'time_-1' in time_dummies.columns:
    time_dummies.drop(columns=['time_-1'], inplace=True)

# 组装回归所需数据结构：被解释变量、时间虚拟变量、控制变量、固定效应虚拟变量
df_pt = pd.concat([df_treated['诉求量_log'], time_dummies, df_treated[controls], pd.get_dummies(df_treated[['地级市', '年份_final']], drop_first=True, dtype=int)], axis=1)

# 构建模型变量并加入常数项
X_pt = sm.add_constant(df_pt.drop(columns=['诉求量_log']))
Y_pt = df_pt['诉求量_log']

# 考虑到随机误差项的潜在相关性，使用聚类稳健标准误（聚类到地级市层面）
model_pt = sm.OLS(Y_pt, X_pt).fit(cov_type='cluster', cov_kwds={'groups': df_treated['地级市']})

# 整理时间虚拟变量的系数、P值及置信区间
pt_results = pd.DataFrame({
    'Coef': model_pt.params,
    'P-value': model_pt.pvalues,
    'CI Lower (2.5%)': model_pt.conf_int()[0],
    'CI Upper (97.5%)': model_pt.conf_int()[1]
})

# 仅筛选展示代表平行趋势的时间虚拟变量结果
pt_results = pt_results[pt_results.index.str.contains('time_')]
#pt_results.to_excel("4_平行趋势检验结果.xlsx")

print("所有数据处理与模型检验完毕，四个Excel文件已生成。")

In [78]:
import pandas as pd
df = pd.read_excel(r'E:\杂七杂八\2026 tjjm\2_处理后含控制变量数据.xlsx')
y_col = '诉求量_log'
df[y_col] = np.log1p(df['诉求量'])

# B. 处理时间固定效应 (年份在第E列，即索引4)
year_col_name = df.columns[4]
print(f"检测到年份列: {year_col_name}")
year_dummies = pd.get_dummies(df[year_col_name], prefix='year', drop_first=True)
year_dummy_cols = year_dummies.columns.tolist()

# C. 处理地区变量 (地区在第A列/省份)
region_col_name = df.columns[0]
print(f"检测到地区列: {region_col_name}")
region_column = df.columns[0]
region_dummies = pd.get_dummies(df[region_column])
region_dummy_col = region_dummies.columns.to_list()
t_col = 'post（开放数据平台时间虚拟变量）'
controls = ['地区生产总值(万元)','人口密度(人／平方公里)', '第三产业增加值占GDP比重(%)', '人力资本水平', '每百人公共图书馆藏书(册、件)',
            '传统基础设施', '互联网发展水平', '金融发展水平', '财政压力水平', '科学支出水平','城市发展水平','外商投资水平','生活垃圾无害化处理率(%)','交通便捷程度']
# 最终控制变量 = 自定义变量 + 时间固定效应虚拟变量（保留原有的时间效应）
x_cols = controls + year_dummy_cols
df_final = pd.concat([df, year_dummies], axis=1)
df_control2 =  pd.concat([df,year_dummies,region_dummies],axis = 1)

cols_to_use = [y_col, t_col, region_col_name] + x_cols
df_dml = df_final.dropna(subset=cols_to_use).copy()

检测到年份列: 年份_final
检测到地区列: 省份_final


In [49]:
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
model_1 = RandomForestRegressor(n_estimators=50, max_depth=6, n_jobs=-1, random_state=123)
model_2 = RandomForestClassifier(n_estimators=50, max_depth=6, n_jobs=-1, random_state=123)
Y_all = df_dml[y_col].values
T_all = df_dml[t_col].values
X_all = df_dml[x_cols].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

In [54]:
from tabulate import tabulate
from econml.dml import LinearDML 
est =  LinearDML(model_y = model_1,model_t = model_2,discrete_treatment = True,cv = 5 ,random_state = 456)
est.fit(Y_all, T_all, X=X_all_scaled, W=None)
ate = est.const_marginal_ate(X_all_scaled).item()
summary_table = est.summary().tables[1]
p_val = float(summary_table.data[1][4])
stderr = float(summary_table.data[1][2])
# 标记显著性星号
stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
global_results = []
global_results.append(["randomforest", f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])
print(tabulate(global_results, headers=["模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))

+--------------+-----------------+----------+-------+
| 模型组合     | ATE (log效应)   |   标准误 |   P值 |
+==============+=================+==========+=======+
| randomforest | 0.2784***       |    0.087 | 0.001 |
+--------------+-----------------+----------+-------+


调整研究样本，
考虑到中国省市间宽带建设基础存在较大差距，采用所有地级及以上城市进行回归分析
可能导致估计偏误，因此，本文在 282 个地级及以上城市中剔除宽带建设基础较差的七个省
份的城市，包括甘肃、青海、宁夏、新疆、西藏、云南以及贵州，并剔除发展基础较好的四
个直辖市，包括北京、天津、上海以及重庆，保留其他城市样本进行回归分析。

In [64]:
region_remove =  [
    '甘肃省', '青海省', '宁夏回族自治区', '新疆维吾尔自治区', 
    '西藏自治区', '云南省', '贵州省',
    '北京市', '天津市', '上海市', '重庆市'
]
dml_remove = df_dml[~df_dml['省份_final'].isin(region_remove)]

In [68]:
from tabulate import tabulate
from econml.dml import LinearDML 
Y_all = dml_remove[y_col].values
T_all = dml_remove[t_col].values
X_all = dml_remove[x_cols].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)
est =  LinearDML(model_y = model_1,model_t = model_2,discrete_treatment = True,cv = 5 ,random_state = 456)
est.fit(Y_all, T_all, X=X_all_scaled, W=None)
ate = est.const_marginal_ate(X_all_scaled).item()
summary_table = est.summary().tables[1]
p_val = float(summary_table.data[1][4])
stderr = float(summary_table.data[1][2])
# 标记显著性星号
stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
global_results = []
global_results.append(["randomforest", f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])
print(tabulate(global_results, headers=["模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))

+--------------+-----------------+----------+-------+
| 模型组合     | ATE (log效应)   |   标准误 |   P值 |
+==============+=================+==========+=======+
| randomforest | 0.3636***       |    0.083 |     0 |
+--------------+-----------------+----------+-------+


进行省份年份的固定效应

In [80]:
from sklearn.preprocessing import OrdinalEncoder
df_dml['province_year_interaction'] = df_dml[region_col_name].astype(str) + '_' + df_dml[year_col_name].astype(str)
# 生成交互虚拟变量（注意：会产生很多虚拟变量）
encoder = OrdinalEncoder()
df_dml['province_year_interaction'] =  encoder.fit_transform(df_dml[['province_year_interaction']])
df_dml_combine = df_dml.copy()

In [84]:
from tabulate import tabulate
from econml.dml import LinearDML 
x_cols_combine = x_cols + ['province_year_interaction']
Y_all = df_dml_combine[y_col].values
T_all = df_dml_combine[t_col].values
X_all = df_dml_combine[x_cols_combine].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)
est =  LinearDML(model_y = model_1,model_t = model_2,discrete_treatment = True,cv = 5 ,random_state = 456)
est.fit(Y_all, T_all, X=X_all_scaled, W=None)
ate = est.const_marginal_ate(X_all_scaled).item()
summary_table = est.summary().tables[1]
p_val = float(summary_table.data[1][4])
stderr = float(summary_table.data[1][2])
# 标记显著性星号
stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
global_results = []
global_results.append(["randomforest", f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])
print(tabulate(global_results, headers=["模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))

+--------------+-----------------+----------+-------+
| 模型组合     | ATE (log效应)   |   标准误 |   P值 |
+==============+=================+==========+=======+
| randomforest | 0.2956***       |    0.093 | 0.001 |
+--------------+-----------------+----------+-------+


由于回归样本中的异常值可能导致估计结果有偏，特别是通过组合赋权法合成的城市包
容性绿色增长指数，因此，本文将基准回归中除处置变量外的所有变量均进行了 1% 、99%
分位点以及 5% 、95% 分位点的缩尾处理，将高于最高分位点和低于最低分位点的数值进行
替换，并以此进行回归分析。

In [85]:
def winsorize_series(series, lower_quantile, upper_quantile):
    """
    对单个序列进行缩尾处理
    """
    lower_bound = series.quantile(lower_quantile)
    upper_bound = series.quantile(upper_quantile)
    return series.clip(lower=lower_bound, upper=upper_bound)

def winsorize_dataframe(df, treatment_col, controls, y_col, lower_quantile, upper_quantile):
    """
    对数据框进行缩尾处理
    df: 原始数据框
    treatment_col: 处置变量列名（不进行缩尾）
    controls: 控制变量列表
    y_col: 因变量列名
    lower_quantile: 下分位数（如0.01代表1%）
    upper_quantile: 上分位数（如0.99代表99%）
    """
    df_winsorized = df.copy()
    
    # 需要缩尾的变量：因变量 + 控制变量
    vars_to_winsorize = [y_col] + controls
    
    for var in vars_to_winsorize:
        if var in df.columns:
            df_winsorized[var] = winsorize_series(df[var], lower_quantile, upper_quantile)
            print(f"  已缩尾 {var}: 下界={df[var].quantile(lower_quantile):.4f}, 上界={df[var].quantile(upper_quantile):.4f}")
    
    # 处置变量保持不变
    print(f"  处置变量 {treatment_col} 未进行缩尾处理")
    
    return df_winsorized

# ============ 应用到你已有的数据 ============

# 假设你已经有了 df_dml 或 df_final
# 定义变量
treatment_col = 'post（开放数据平台时间虚拟变量）'  # 处置变量，不缩尾
y_col = '诉求量_log'  # 因变量
controls = ['地区生产总值(万元)','人口密度(人／平方公里)', '第三产业增加值占GDP比重(%)', 
            '人力资本水平', '每百人公共图书馆藏书(册、件)', '传统基础设施', '互联网发展水平', 
            '金融发展水平', '财政压力水平', '科学支出水平','城市发展水平','外商投资水平',
            '生活垃圾无害化处理率(%)','交通便捷程度']

# ============ 方案1：1%-99%缩尾 ============
print("=" * 50)
print("进行 1%-99% 缩尾处理...")
df_winsorized_1_99 = winsorize_dataframe(
    df=df_dml,  # 或 df_final
    treatment_col=treatment_col,
    controls=controls,
    y_col=y_col,
    lower_quantile=0.01,
    upper_quantile=0.99
)

进行 1%-99% 缩尾处理...
  已缩尾 诉求量_log: 下界=0.0000, 上界=5.3471
  已缩尾 地区生产总值(万元): 下界=1743027.0800, 上界=200467800.0000
  已缩尾 人口密度(人／平方公里): 下界=18.0000, 上界=1343.7000
  已缩尾 第三产业增加值占GDP比重(%): 下界=20.6156, 上界=71.6044
  已缩尾 人力资本水平: 下界=7.4152, 上界=1180.3234
  已缩尾 每百人公共图书馆藏书(册、件): 下界=6.0000, 上界=520.0000
  已缩尾 传统基础设施: 下界=6.6161, 上界=10.8487
  已缩尾 互联网发展水平: 下界=9.8460, 上界=15.2525
  已缩尾 金融发展水平: 下界=0.2971, 上界=3.3418
  已缩尾 财政压力水平: 下界=0.9953, 上界=11.1934
  已缩尾 科学支出水平: 下界=7.2714, 上界=14.4767
  已缩尾 城市发展水平: 下界=8.9635, 上界=12.0677
  已缩尾 外商投资水平: 下界=0.0000, 上界=7.1998
  已缩尾 生活垃圾无害化处理率(%): 下界=19.8668, 上界=100.0000
  已缩尾 交通便捷程度: 下界=0.3877, 上界=7.1881
  处置变量 post（开放数据平台时间虚拟变量） 未进行缩尾处理


In [96]:
from tabulate import tabulate
from econml.dml import LinearDML 
Y_all = df_winsorized_1_99[y_col].values
T_all = df_winsorized_1_99[t_col].values
X_all = df_winsorized_1_99[x_cols].values

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)
est =  LinearDML(model_y = model_1,model_t = model_2,discrete_treatment = True,cv = 5 ,random_state = 78)
est.fit(Y_all, T_all, X=X_all_scaled, W=None)
ate = est.const_marginal_ate(X_all_scaled).item()
summary_table = est.summary().tables[1]
p_val = float(summary_table.data[1][4])
stderr = float(summary_table.data[1][2])
# 标记显著性星号
stars = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
global_results = []
global_results.append(["randomforest", f"{ate:.4f}{stars}", f"{stderr:.4f}", f"{p_val:.4f}"])
print(tabulate(global_results, headers=["模型组合", "ATE (log效应)", "标准误", "P值"], tablefmt="grid"))

+--------------+-----------------+----------+-------+
| 模型组合     | ATE (log效应)   |   标准误 |   P值 |
+==============+=================+==========+=======+
| randomforest | 0.1705**        |    0.086 | 0.048 |
+--------------+-----------------+----------+-------+
